# paper-graph-rag · 逐步探索

这个 notebook 把 `src/` 里的四个脚本拆成可逐个运行的单元，用来看清每一步在做什么、
中间产物长什么样。**它不是替代源码，是源码的放大镜。**

真正跑整条 pipeline 还是用命令行（在项目根目录）：

```
python src/01_fetch.py       # 抓取语料（联网，会覆盖 data/raw/works_raw.json）
python src/02_eval_set.py    # 构造评估集
python src/evaluate.py       # 跑评测，输出对比矩阵
```

> 启动前注意：内核选 **pytorch**（`D:\ancodoa\envs\pytorch`），
> 只有这个环境装了 scikit-learn。默认的 python3 内核不是它。

## 0 · 环境准备

自动定位项目根目录，并把 `src/` 加进模块搜索路径 —— 之后就能直接 `import retriever`。

In [ ]:
import sys, os, json
from pathlib import Path

MARKER = Path("src") / "retriever.py"


def find_root():
    cands = [Path.cwd(), Path.home() / "Desktop" / "paper-graph-rag"]
    for base in list(Path.cwd().parents)[:3]:
        cands += [base / "paper-graph-rag", base / "Desktop" / "paper-graph-rag"]
    for c in cands:
        if (c / MARKER).exists():
            return c.resolve()
    return None


PROJ = find_root()
assert PROJ is not None, "没自动找到项目根目录 —— 手动把 PROJ 改成 paper-graph-rag 的绝对路径"

os.chdir(PROJ)
sys.path.insert(0, str(PROJ / "src"))

import sklearn, requests
print("项目根目录:", PROJ)
print("Python :", sys.version.split()[0])
print("sklearn:", sklearn.__version__, "| requests:", requests.__version__)

## 1 · 语料 —— 382 篇论文长什么样

`Corpus` 读取 `data/raw/works_raw.json`（`01_fetch.py` 的产物，已随仓库提交）。

In [ ]:
from retriever import Corpus

corpus = Corpus()
print(f"语料规模  : {len(corpus)} 篇")
print(f"年份跨度  : {min(w['year'] for w in corpus.works)} – {max(w['year'] for w in corpus.works)}")

years = {}
for w in corpus.works:
    years[w["year"]] = years.get(w["year"], 0) + 1
print("年份分布  :", dict(sorted(years.items())))

In [ ]:
# 拆开看一篇，确认字段
w = corpus.works[0]
print("可用字段:", list(w.keys()))
print()
for k in ["id", "title", "year", "venue", "cited_by_count", "tags"]:
    print(f"{k:16}: {w[k]}")
print(f"{'referenced_works':16}: {len(w['referenced_works'])} 条（大部分在语料库之外）")
print()
print("摘要前 400 字:")
print(w["abstract"][:400], "...")

In [ ]:
# 引用图：这就是整个项目的核心数据结构
from collections import Counter

edges = [(w["id"], r)
         for w in corpus.works
         for r in w["referenced_works"]
         if r in corpus.inside]

print(f"库内引用边          : {len(edges)} 条")
print(f"有库内出度的论文    : {len({a for a, b in edges})} 篇")

indeg = Counter(b for a, b in edges)
print()
print("被库内引用最多的 5 篇（入度 = 谁会出现在引用型答案里）:")
for wid, n in indeg.most_common(5):
    print(f"  {n:3} 次  {corpus.title[wid][:72]}")

print()
print("注：边只有 191 条，因为语料只覆盖 2022–2026，很多论文引用的是库外的老论文。")
print("    这个稀疏性是真实约束，不是 bug —— README 的「已知局限」里写了。")

## 2 · 评估集 —— 75 题是怎么来的

`02_eval_set.py` 生成三类题，**每一类考察不同的检索能力**：

| 类型 | 题数 | query 怎么造 | gold |
|---|---|---|---|
| keyword | 30 | 标题去停用词后的关键词 | 该论文 |
| semantic | 20 | **人工改写**的自然语言描述，刻意避开标题用词 | 该论文 |
| citation | 25 | 「哪些论文引用了 X」 | 库内引用 X 的论文集合 |

In [ ]:
items = json.loads((PROJ / "data" / "eval_set.json").read_text(encoding="utf-8"))
print("评估集:", len(items), "题 |", dict(Counter(i["type"] for i in items)))

for t in ["keyword", "semantic", "citation"]:
    ex = next(i for i in items if i["type"] == t)
    print()
    print(f"— {t} —")
    print("  query:", ex["query"][:120])
    print("  gold :", len(ex["gold"]), "篇")
    if t != "citation":
        print("  对应论文:", corpus.title[ex["gold"][0]][:80])

print()
print("关键在 semantic 那 20 条：如果 query 直接用标题，就会退化成关键词匹配，")
print("测不出语义检索的价值。所以全部用不同措辞重写了一遍。")

## 3 · 亲手跑一次检索 —— 看到差距是怎么拉开的

两个检索器共用同一个接口 `search(query, k) -> [doc_id, ...]`，
所以能放在同一套评估集上横向比。

In [ ]:
from retriever import build

tfidf = build("tfidf", corpus)
graph = build("graph", corpus)
print("已就绪:", tfidf.name, "/", graph.name)
print()
print("TfidfRetriever : sublinear tf + 1-2 gram 的 TF-IDF 余弦相似度")
print("GraphRetriever : 从 query 里解析标题做实体链接 → 沿引用图入边展开")

In [ ]:
# ── 题 1：关键词型 ── 词法检索的主场
q = next(i for i in items if i["type"] == "keyword")
print("query :", q["query"])
print("gold  :", corpus.title[q["gold"][0]][:80])
print()

for tag, r in [("TF-IDF", tfidf), ("图检索", graph)]:
    hit = r.search(q["query"], 5)
    print(f"{tag} → 命中: {q['gold'][0] in hit}")
    for i, d in enumerate(hit, 1):
        print(f"   {i}. {corpus.title[d][:70]}")
    if not hit:
        print("    (返回空)")
    print()

In [ ]:
# ── 题 2：语义型 ── 同一个检索器，立刻崩掉
q = next(i for i in items if i["type"] == "semantic")
print("query :", q["query"])
print("gold  :", corpus.title[q["gold"][0]][:80])
print()

hit = tfidf.search(q["query"], 5)
print("TF-IDF top5 命中:", q["gold"][0] in hit)
for i, d in enumerate(hit, 1):
    print(f"   {i}. {corpus.title[d][:70]}")
if not hit:
    print("    (返回空)")

print("""
为什么崩？query 是人工改写的人话，刻意避开了标题用词 —— 和论文原文没有词汇重叠。
TF-IDF 靠词面匹配，匹配不上就排不到前面。
同一个检索器、同一份语料，只换了查询的表述方式，HR@5 就从 1.000 掉到 0.350。
这不是「效果差一点」，是查询分布一变就失效 —— 和 OOD 泛化是同一个问题框架。
""")

In [ ]:
# ── 题 3：引用型 ── 词法检索完全答不了的问题
q = next(i for i in items if i["type"] == "citation")
print("query :", q["query"][:110])
print("gold  :", len(q["gold"]), "篇（库内引用它的论文集合）")
print()

for tag, r in [("TF-IDF", tfidf), ("图检索", graph)]:
    hit = r.search(q["query"], 10)
    inter = set(hit) & set(q["gold"])
    print(f"{tag}: top10 返回 {len(hit)} 条 | 命中 gold {len(inter)}/{len(q['gold'])}")
    for d in hit[:5]:
        flag = "[命中]" if d in q["gold"] else "[    ]"
        print(f"   {flag} {corpus.title[d][:66]}")
    if not hit:
        print("    (返回空)")
    print()

print("答案不在任何单篇文档的文本里，而在文档之间的边上 —— 所以向量检索同样答不了。")
print("这是建图的直接理由，也是唯一一条「能力有无」而非「程度高低」的差异。")

## 4 · 全指标矩阵

`evaluate.py` 让**每个检索器跑全部三类题**（不是给每类题配它擅长的检索器），
只有这样得到的矩阵才能回答问题：图结构到底解决了什么、又解决不了什么。

In [ ]:
report = json.loads((PROJ / "results" / "metrics.json").read_text(encoding="utf-8"))

print(f"语料 {report['corpus_size']} 篇 | 评估集 {report['eval_size']} 题 | K={report['k']}")
print()
print(f"{'检索器':<9}{'题型':<11}{'题数':>5}{'HR@1':>8}{'HR@5':>8}{'MRR@10':>9}{'Recall@10':>11}")
print("-" * 62)
for name, types in report["results"].items():
    for t, a in types.items():
        print(f"{name:<9}{t:<11}{a['n']:>5}{a['hr1']:>8.3f}{a['hr5']:>8.3f}"
              f"{a['mrr']:>9.3f}{a['recall10']:>11.3f}")
    print("-" * 62)

print("""
三条数字对应三条论证链：

  1. 关键词明确时，词法检索已经够用          HR@5 = 1.000
  2. 换成自然语言表述，词法检索立刻崩掉      HR@5 = 0.350
  3. 关系型问题词法检索根本答不了           MRR 0.139  vs  图检索 1.000

第 3 条是建图的直接理由。
""")

## 附 · 可选：抓取这一步在调什么 API

对应 `src/01_fetch.py`。只抓 3 条演示，**不会覆盖 `data/raw/works_raw.json`**。

In [ ]:
API = "https://api.openalex.org/works"
params = {
    "filter": ("title_and_abstract.search:graph retrieval augmented generation,"
               "from_publication_date:2022-01-01,is_paratext:false"),
    "per-page": 3,
    "sort": "relevance_score:desc",
    "select": "id,title,abstract_inverted_index,publication_year,"
              "referenced_works,cited_by_count,type",
}
r = requests.get(API, params=params, timeout=45)
data = r.json()
print("HTTP", r.status_code, "| 这个 query 的命中总量:", data["meta"]["count"])
print("（这个数字每次查都会变 —— OpenAlex 的库在滚动更新，README 记的是快照当时的 4597）")
print()

for w in data["results"]:
    print(f"{w['id']}  {w['publication_year']}  cited={w['cited_by_count']}")
    print("   ", w["title"][:100])
    print("    倒排索引词数:", len(w.get("abstract_inverted_index") or {}))

print("""
注意 abstract_inverted_index：OpenAlex 把摘要存成 {词: [位置, ...]}，
需要按位置还原成文本才能检索 —— 见 01_fetch.py 的 restore_abstract()。

为什么用 OpenAlex 不用 arXiv：arXiv 共享出口实测 HTTP 429 限流，
而且没有引用关系字段；OpenAlex 的 referenced_works 直接白送一张引用图。
""")